# Pipeline de datos: construcción del corpus y procesamiento

Recorrido documentado del pipeline que sustenta los resultados **R1.1** y **R1.2**.

| Resultado | Medio de verificación comprometido | Indicador |
| --- | --- | --- |
| R1.1 | Repositorio digital documentado con estructura clara de datos crudos y procesados. Informe descriptivo del tratamiento de los datasets. | Al menos 25% de las muestras con las tres modalidades y 40% con dos; prevalencia entre 0.40 y 0.60; formato estándar listo para ingesta; prueba de humo automatizada. |
| R1.2 | Código fuente y notebook documentado. | Ejecución sin errores del script de limpieza y extracción de características del 90% de las muestras. |

El cuaderno funciona en dos modos. Si el corpus está construido en
`data/Datasets_Procesados/Dataset_Real.parquet`, mide sobre él. Si no lo está, lee
las cifras del informe que dejó la última ejecución en
`medios_de_verificacion/experimentos/e0/e0.json`, de modo que puede leerse sin
descargar 441 MB de correo.

In [ ]:
import json
import sys
from pathlib import Path

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE / "src"))

CORPUS = BASE / "data" / "Datasets_Procesados" / "Dataset_Real.parquet"
INFORME_E0 = BASE / "medios_de_verificacion" / "experimentos" / "e0" / "e0.json"

hay_corpus = CORPUS.exists()
print("corpus construido:", hay_corpus)
print("informe de la última ejecución:", INFORME_E0.exists())

## 1. Datos crudos: de dónde sale cada colección

Ocho colecciones públicas. La distinción que gobierna todo el trabajo es el
**formato de publicación**: cinco se publican como mensaje original (`.eml` o buzones
`mbox`) y conservan el marcado del cuerpo y las cabeceras de tránsito; tres se
publican en formato tabular y solo conservan el texto.

Solo de las primeras pueden extraerse la estructura del documento y los metadatos de
red, que son las dos modalidades no textuales de la arquitectura.

| Colección | Formato de origen | Clase | Módulo de descarga |
| --- | --- | --- | --- |
| phishing_pot | `.eml` (repositorio git) | phishing | `downloaders/correo_real.py` |
| Nazario | `mbox` | phishing | `downloaders/correo_real.py` |
| SpamAssassin | `mbox` (`.tar.bz2`) | legítimo | `downloaders/correo_real.py` |
| Fedora | `mbox` de lista de discusión | legítimo | `downloaders/listas_correo.py` |
| kernel_lists | `mbox` de lista de discusión | legítimo | `downloaders/listas_correo.py` |
| Kaggle | CSV | ambas | `downloaders/kaggle_phishing.py` |
| CEAS_08 | JSONL | legítimo | `downloaders/phish_mmf.py` |
| datacon2023 | JSONL | phishing | `downloaders/phish_mmf.py` |

La descarga no se ejecuta aquí porque tarda y necesita red. El comando es:

```bash
python -m phishing_pipeline.downloaders.correo_real
python -m phishing_pipeline.downloaders.listas_correo
```

## 2. De mensaje crudo a fila del corpus

`phishing_pipeline.cleaners.correo_crudo` analiza cada mensaje conforme al RFC 5322 y
extrae, de una sola pasada, las tres modalidades:

1. **Texto**: cuerpo en texto plano; si el mensaje solo trae HTML, se convierte.
2. **Estructura**: estadísticas del árbol DOM del cuerpo (`features/dom_stats.py`):
   número de enlaces, formularios, imágenes, profundidad del árbol, presencia de
   `<script>`.
3. **Red**: características léxicas de las URL y resultado de autenticación leído de
   las cabeceras `Authentication-Results` (`features/network.py`): SPF, DKIM, DMARC.

Una modalidad ausente no es un cero: se marca con su **bandera de disponibilidad**, y
esa distinción es la que permite después auditar si la mera disponibilidad predice la
clase.

In [ ]:
import tempfile
from pathlib import Path as _Path

from phishing_pipeline.cleaners.correo_crudo import limpiar_correo_crudo

EJEMPLO = '''From: Soporte <aviso@banco-seguro.example>
To: cliente@example.com
Subject: Verificacion de su cuenta
Authentication-Results: mx.example.com; spf=fail; dkim=none; dmarc=fail
Content-Type: text/html

<html><body><p>Su cuenta sera suspendida.</p>
<a href="http://192.0.2.10/verificar">Verificar ahora</a>
<form action="http://192.0.2.10/robar"><input name="clave"></form>
</body></html>
'''

with tempfile.TemporaryDirectory() as tmp:
    ruta = _Path(tmp) / 'ejemplo.eml'
    ruta.write_text(EJEMPLO, encoding='utf-8')
    df = limpiar_correo_crudo(ruta, fuente='ejemplo', etiqueta=1)

fila = df.iloc[0]
for campo in df.columns:
    valor = fila[campo]
    if isinstance(valor, str) and len(valor) > 80:
        valor = valor[:80] + '...'
    print(f'{campo:32s} {valor}')


## 3. Deduplicación y agrupamiento por plantilla

Las campañas de phishing envían miles de mensajes casi idénticos. Si dos variantes de
la misma campaña caen una en entrenamiento y otra en prueba, el modelo memoriza la
plantilla y la métrica sube sin que nada lo advierta.

`phishing_pipeline.dedup.near_duplicates` agrupa por similitud con MinHash y LSH, y
asigna a cada mensaje un `template_cluster_id`. La partición reparte **conglomerados
completos** y no mensajes sueltos.

El experimento E4 mide cuánto infla la métrica no hacerlo, que es el valor
metodológico propio de esta decisión.

## 4. Estructura de los datos procesados

El corpus se materializa como un único `.parquet`, que es el formato estándar listo
para ingesta que exige el indicador de R1.1.

In [ ]:
from phishing_pipeline.schema import CANONICAL_COLUMNS

print('data/')
print('  Datasets_Originales/        mensajes tal como se publican (no versionado)')
print('  Datasets_Procesados/')
print('    Dataset_Real.parquet      el corpus, una fila por mensaje')
print('    splits_real_group_aware/  particiones agrupadas por plantilla')
print('  reports/                    informes de medicion')
print('  predictions/                prediccion fila por fila de cada modelo')

print(f'\nEsquema canonico: {len(CANONICAL_COLUMNS)} columnas')
for c in CANONICAL_COLUMNS:
    print('  ', c)

if hay_corpus:
    import pandas as pd

    corpus = pd.read_parquet(CORPUS)
    print(f'\n{len(corpus):,} filas y {len(corpus.columns)} columnas en el corpus')
else:
    print('\nEl corpus no esta construido en esta copia.')
    print('Construirlo con: python -m phishing_pipeline.corpus_real')


## 5. Contraste del indicador de R1.1

Los cuatro criterios del indicador, cada uno con su umbral y el valor medido.

In [ ]:
e0 = json.loads(INFORME_E0.read_text(encoding="utf-8"))
ind = e0["indicador_r1_1"]

print(f"{'Criterio':52s} {'Umbral':12s} {'Medido':10s} Cumple")
for c in ind["criterios"]:
    print(f"{c['criterio']:52s} {c['umbral']:12s} {c['medido']:10s} "
          f"{'sí' if c['cumple'] else 'NO'}")

corpus_info = e0["corpus"]
print(f"\nCorpus: {corpus_info['n']:,} mensajes, prevalencia {corpus_info['prevalencia']:.4f}")
print(f"Trimodal: {ind['trimodal']['n']:,} ({ind['trimodal']['porcentaje']:.2f}%)")
print(f"Bimodal:  {ind['bimodal']['n']:,} ({ind['bimodal']['porcentaje']:.2f}%)")

In [ ]:
print(f"{'Colección':18s} {'Mensajes':>10s} {'Legítimos':>10s} {'Phishing':>10s}")
for nombre, d in sorted(corpus_info["por_fuente"].items(), key=lambda x: -x[1]["n"]):
    print(f"{nombre:18s} {d['n']:>10,} {d['legitimos']:>10,} {d['phishing']:>10,}")

## 6. Prueba de humo: indicador de R1.2

Ocho comprobaciones sobre el conjunto construido, antes de que ningún modelo lo use.
Comprueban integridad y, en particular, que la cobertura textual sea completa: es la
cifra contra la que se contrasta el umbral del 90% de muestras procesadas sin error.

In [ ]:
humo = e0["prueba_de_humo"]
for c in humo["comprobaciones"]:
    print(f"{c['resultado']:4s}  {c['comprobacion']:56s} {c['detalle']}")
print(f"\n{len(humo['comprobaciones'])} comprobaciones")

## 7. Auditoría de fuga por disponibilidad de campo

El indicador de R1.1 exige descartar de forma explícita que el modelo pueda resolver
la tarea por un atajo. El riesgo concreto es este: si todos los mensajes de una clase
vinieran de colecciones que conservan HTML y los de la otra no, bastaría preguntarse
si el campo existe.

`phishing_pipeline.auditoria_fugas` entrena un clasificador que ve **solo** las
banderas de disponibilidad, sin acceso al contenido. El experimento E5 lo lleva más
lejos y mide, característica por característica, cuánta información aporta sobre la
clase frente a cuánta aporta sobre la colección de origen.

In [ ]:
fugas = e0["auditoria_de_fugas_r1_2"]
print(json.dumps(fugas, ensure_ascii=False, indent=2)[:1200])

## 8. Reproducir el pipeline entero

```bash
python -m phishing_pipeline.downloaders.correo_real     # phishing_pot, Nazario, SpamAssassin
python -m phishing_pipeline.downloaders.listas_correo   # Fedora, kernel_lists
python -m phishing_pipeline.downloaders.kaggle_phishing # Kaggle
python -m phishing_pipeline.downloaders.phish_mmf       # CEAS_08, datacon2023
python -m phishing_pipeline.corpus_real                 # construye Dataset_Real.parquet
python scripts/experimentos/e0_corpus_y_pipeline.py     # mide el indicador y la prueba de humo
```

El corpus resultante no se versiona: son 441 MB, por encima del límite de 100 MB por
archivo de la plataforma. Se reconstruye de forma íntegra con los comandos anteriores,
y su huella criptográfica queda registrada en la procedencia de cada informe, de modo
que puede comprobarse que la reconstrucción coincide con la que produjo los
resultados.